In [2]:
## Data Collection

import nltk
nltk.download('gutenberg')
from nltk.corpus import gutenberg
import pandas as pd

## Load the dataset
data = gutenberg.raw('shakespeare-hamlet.txt')
## save to a file
with open('hamlet.txt','w') as file:
    file.write(data)

[nltk_data] Downloading package gutenberg to
[nltk_data]     C:\Users\subha\AppData\Roaming\nltk_data...
[nltk_data]   Package gutenberg is already up-to-date!


In [5]:
# Data Preprocessing

import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

##Load the dataset
with open('hamlet.txt','r') as file:
    text = file.read().lower()

## Tokenize
tokenizer = Tokenizer()
tokenizer.fit_on_texts([text])

total_words = len(tokenizer.word_index) + 1
total_words

4818

In [7]:
tokenizer.word_counts

OrderedDict([('the', 993),
             ('tragedie', 4),
             ('of', 610),
             ('hamlet', 100),
             ('by', 105),
             ('william', 1),
             ('shakespeare', 1),
             ('1599', 1),
             ('actus', 2),
             ('primus', 1),
             ('scoena', 1),
             ('prima', 1),
             ('enter', 85),
             ('barnardo', 8),
             ('and', 862),
             ('francisco', 2),
             ('two', 22),
             ('centinels', 1),
             ("who's", 2),
             ('there', 76),
             ('fran', 8),
             ('nay', 26),
             ('answer', 9),
             ('me', 228),
             ('stand', 15),
             ('vnfold', 3),
             ('your', 253),
             ('selfe', 68),
             ('bar', 7),
             ('long', 17),
             ('liue', 15),
             ('king', 171),
             ('he', 196),
             ('you', 522),
             ('come', 104),
             ('most', 77),
  

In [11]:
## Input sequences
input_sequences = []
for line in text.split('\n'):
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

In [12]:
input_sequences

[[1, 687],
 [1, 687, 4],
 [1, 687, 4, 45],
 [1, 687, 4, 45, 41],
 [1, 687, 4, 45, 41, 1886],
 [1, 687, 4, 45, 41, 1886, 1887],
 [1, 687, 4, 45, 41, 1886, 1887, 1888],
 [1180, 1889],
 [1180, 1889, 1890],
 [1180, 1889, 1890, 1891],
 [57, 407],
 [57, 407, 2],
 [57, 407, 2, 1181],
 [57, 407, 2, 1181, 177],
 [57, 407, 2, 1181, 177, 1892],
 [407, 1182],
 [407, 1182, 63],
 [408, 162],
 [408, 162, 377],
 [408, 162, 377, 21],
 [408, 162, 377, 21, 247],
 [408, 162, 377, 21, 247, 882],
 [18, 66],
 [451, 224],
 [451, 224, 248],
 [451, 224, 248, 1],
 [451, 224, 248, 1, 30],
 [408, 407],
 [451, 25],
 [408, 6],
 [408, 6, 43],
 [408, 6, 43, 62],
 [408, 6, 43, 62, 1893],
 [408, 6, 43, 62, 1893, 96],
 [408, 6, 43, 62, 1893, 96, 18],
 [408, 6, 43, 62, 1893, 96, 18, 566],
 [451, 71],
 [451, 71, 51],
 [451, 71, 51, 1894],
 [451, 71, 51, 1894, 567],
 [451, 71, 51, 1894, 567, 378],
 [451, 71, 51, 1894, 567, 378, 80],
 [451, 71, 51, 1894, 567, 378, 80, 3],
 [451, 71, 51, 1894, 567, 378, 80, 3, 273],
 [451, 71

In [ ]:
# Pad Sequence
max_sequence_len = max(len(x) for x in input_sequences)
max_sequence_len

14

In [14]:
input_sequences=np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

In [15]:
## Create predictors and label

import tensorflow as tf
x,y=input_sequences[:,:-1], input_sequences[:,-1]
y=tf.keras.utils.to_categorical(y, num_classes=total_words)

In [16]:
# Train test split
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

In [21]:
## Train our LSTM RNN
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding,LSTM,Dense,Dropout

## Define model
model = Sequential()
model.add(Embedding(total_words,100,input_length=max_sequence_len-1))
model.add(LSTM(150,return_sequences=True))
model.add(Dropout(0.2))
model.add(LSTM(100))
model.add(Dense(total_words, activation="softmax")) # we are using softmax because our output is multiclass

## Compile the model
model.compile(loss="categorical_crossentropy", optimizer='adam', metrics=['accuracy'])
model.summary()

Model: "sequential_3"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 embedding_3 (Embedding)     (None, 13, 100)           481800    
                                                                 
 lstm_4 (LSTM)               (None, 13, 150)           150600    
                                                                 
 dropout_2 (Dropout)         (None, 13, 150)           0         
                                                                 
 lstm_5 (LSTM)               (None, 100)               100400    
                                                                 
 dense_2 (Dense)             (None, 4818)              486618    
                                                                 
Total params: 1219418 (4.65 MB)
Trainable params: 1219418 (4.65 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [22]:
## Train the model

history=model.fit(x_train, y_train, epochs=50, validation_data=(x_test,y_test),verbose=1)

Epoch 1/50


644/644 [==============================] - 29s 34ms/step - loss: 6.8983 - accuracy: 0.0332 - val_loss: 6.7492 - val_accuracy: 0.0326
Epoch 2/50
644/644 [==============================] - 25s 39ms/step - loss: 6.4637 - accuracy: 0.0386 - val_loss: 6.8402 - val_accuracy: 0.0425
Epoch 3/50
644/644 [==============================] - 33s 51ms/step - loss: 6.3174 - accuracy: 0.0459 - val_loss: 6.9208 - val_accuracy: 0.0478
Epoch 4/50
644/644 [==============================] - 41s 63ms/step - loss: 6.1655 - accuracy: 0.0511 - val_loss: 6.9159 - val_accuracy: 0.0521
Epoch 5/50
644/644 [==============================] - 41s 64ms/step - loss: 6.0233 - accuracy: 0.0558 - val_loss: 6.9276 - val_accuracy: 0.0581
Epoch 6/50
644/644 [==============================] - 39s 61ms/step - loss: 5.8747 - accuracy: 0.0650 - val_loss: 7.0044 - val_accuracy: 0.0608
Epoch 7/50
644/644 [==============================] - 33s 51ms/step - loss: 5.7222 - accuracy: 0.0715 - val_loss: 7.0775 - val_accurac

In [23]:
## Function to predict the next word
def predict_next_word(model, tokenizer, text, max_sequence_len):
    token_list = tokenizer.texts_to_sequences([text])[0]
    if len(token_list) >= max_sequence_len:
        token_list = token_list[-(max_sequence_len-1):]
    token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
    predicted = model.predict(token_list, verbose=0)
    predicted_word_index = np.argmax(predicted, axis=1)
    for word, index in tokenizer.word_index.items():
        if index == predicted_word_index:
            return word
    return None
        

In [28]:
input_text = 'it is her'
print(f"Input text:{input_text}")
max_sequence_len=model.input_shape[1]+1
next_word=predict_next_word(model, tokenizer, input_text, max_sequence_len)
print(f"Next word Prediction:{next_word}")

Input text:it is her
Next word Prediction:fighting


In [25]:
## save model
model.save("next_word_lstm.h5")
## save tokenizer
import pickle
with open('tokenizer.pickle','wb') as handle:
    pickle.dump(tokenizer, handle, protocol=pickle.HIGHEST_PROTOCOL)

c:\Users\subha\Documents\GitHub\LSTM-RNN-Project\venv\Lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
